<center>
<img src="https://supportvectors.ai/logo-poster-transparent.png" width=400px style="opacity:0.8">
</center>

# Document Ingestion

GraphRAG starts by turning source PDFs into passage-level chunks that preserve local context while staying small enough for LLM extraction.

Set `INGEST_MODE` below to ingest either a single file (`attention.pdf`) or every PDF recursively under `data/`.

In [1]:
%run supportvectors-common.ipynb



<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors. All rights reserved. <blockquote>This notebook is the intellectual property of SupportVectors, and part of its training material. 
Only the participants in SupportVectors workshops are allowed to study the notebooks for educational purposes currently, but is prohibited from copying or using it for any other purposes without written permission.

<b> These notebooks are chapters and sections from Asif Qamar's textbook that he is writing on Data Science. So we request you to not circulate the material to others.</b>
 </blockquote>
 <hr/>
</div>



In [2]:
from memg_concepts.config import DEFAULT_DATA_DIR, DEFAULT_PDF
from memg_concepts.document import (
    build_passages,
    build_passages_from_dir,
    discover_pdfs,
    load_pdf_text,
)

INGEST_MODE = "single"  # "single" for one PDF, "directory" for all PDFs under data/

if INGEST_MODE == "directory":
    pdfs = discover_pdfs(DEFAULT_DATA_DIR)
    print(f"Found {len(pdfs)} PDF(s) under {DEFAULT_DATA_DIR}")
    for pdf in pdfs:
        print(f"  {pdf.relative_to(DEFAULT_DATA_DIR)}")
else:
    raw_text = load_pdf_text(DEFAULT_PDF)
    print(f"Characters loaded: {len(raw_text):,}")
    print(raw_text[:500])

Characters loaded: 39,611
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


In [3]:
if INGEST_MODE == "directory":
    passages = build_passages_from_dir(
        DEFAULT_DATA_DIR,
        chunk_size=800,
        chunk_overlap=120,
        max_chunks_per_doc=12,
    )
else:
    passages = build_passages(
        DEFAULT_PDF,
        chunk_size=800,
        chunk_overlap=120,
        max_chunks=12,
    )

print(f"Passages: {len(passages)}")
for passage in passages[:3]:
    print(passage.passage_id, f"[{passage.source}]", passage.text[:140], "...")

Passages: 12
p_0000 [attention.pdf] Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in ...
p_0001 [attention.pdf] volutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an ...
p_0002 [attention.pdf] translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a  ...


These passages become the **Passage layer** (`M_pas`) in global memory.